# 08 — LangGraph Threads and SQLite Checkpointing

## Goal

Learn how LangGraph persistence works using a SQLite checkpointer.

We will prove that:

- a `thread_id` identifies a persistent agent thread
- messages are stored as part of graph state
- future invocations can continue the same thread
- different thread IDs remain isolated
- state survives Python process restart

This is development persistence only.

Production persistence with Postgres comes later.

In [2]:
from pathlib import Path

db_path = Path("../data/checkpoints.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

print(db_path.resolve())

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\data\checkpoints.db


In [3]:
import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver

connection = sqlite3.connect(
    db_path,
    check_same_thread=False,
)

checkpointer = SqliteSaver(connection)

In [4]:
from deepagents import create_deep_agent

from deep_agents_foundry.model import build_model
from deep_agents_foundry.tools import build_web_search_tool
from deep_agents_foundry.agent import RESEARCH_INSTRUCTIONS


model = build_model()
web_search = build_web_search_tool()

persistent_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=RESEARCH_INSTRUCTIONS,
    checkpointer=checkpointer,
)

C:\Users\shchitt\Downloads\Projects\deep-agents-on-foundry\src\deep_agents_foundry\tools.py:13: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  return WebSearchTool()


In [ ]:
thread_1 = {
    "configurable": {
        "thread_id": "research-thread-001"
    }
}
thread_1

{'configurable': {'thread_id': 'research-thread-001'}}

In [ ]:
result_1 = persistent_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "In one concise paragraph, explain what a Foundry Hosted Agent is."
                ),
            }
        ]
    },
    config=thread_1,
)

print(result_1["messages"][-1].content[-1]['text'])

A **Foundry Hosted Agent** is a **code-based AI agent** that you package as a **container** and deploy to **Microsoft Foundry Agent Service**, where Microsoft provides the managed runtime—secure session-isolated execution, scaling, deployment lifecycle/operations, identity integration, and observability—while your containerized code handles the agent’s orchestration logic and calls models from the Foundry model catalog as needed. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent))


In [8]:
result_2 = persistent_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Now explain the identity part in more detail.",
            }
        ]
    },
    config=thread_1,
)

print(result_2["messages"][-1].content[-1]['text'])

In Foundry Hosted Agents, **identity** means your deployed agent is automatically given its own **dedicated Microsoft Entra ID “agent identity”** at deploy time—implemented as a **service principal**—so the container can authenticate to Foundry resources (models, tools/Toolbox) and other downstream Azure services **without baking secrets into the image**, and so its actions are **separately auditable and governable via Entra/RBAC** like any other workload identity. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)) Additionally, when the agent is invoked from a user-facing surface (for example, Teams/Microsoft 365), Foundry/Toolbox can use **OAuth 2.0 On-Behalf-Of (OBO)** to propagate the **end user’s identity context** to downstream systems when you need the agent to “act as the user,” rather than only as the agent’s own service identity. ([devblogs.microsoft.com](https://devblogs.microsoft.com/foundry/introducing-the-new-hosted-agen

In [9]:
state = persistent_agent.get_state(thread_1)

print(type(state))

<class 'langgraph.types.StateSnapshot'>


In [11]:
print(state.values.keys())

dict_keys(['messages', 'files'])


In [12]:
messages = state.values["messages"]

print("Message count:", len(messages))

for i, message in enumerate(messages):
    print(i, type(message).__name__)

Message count: 4
0 HumanMessage
1 AIMessage
2 HumanMessage
3 AIMessage


In [14]:
from langchain_core.messages import HumanMessage, AIMessage


for message in messages:
    if isinstance(message, HumanMessage):
        print("\nUSER:")
        print(message.content)

    elif isinstance(message, AIMessage):
        print("\nASSISTANT:")
        print(message.content[-1]['text'])


USER:
In one concise paragraph, explain what a Foundry Hosted Agent is.

ASSISTANT:
A **Foundry Hosted Agent** is a **code-based AI agent** that you package as a **container** and deploy to **Microsoft Foundry Agent Service**, where Microsoft provides the managed runtime—secure session-isolated execution, scaling, deployment lifecycle/operations, identity integration, and observability—while your containerized code handles the agent’s orchestration logic and calls models from the Foundry model catalog as needed. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent))

USER:
Now explain the identity part in more detail.

ASSISTANT:
In Foundry Hosted Agents, **identity** means your deployed agent is automatically given its own **dedicated Microsoft Entra ID “agent identity”** at deploy time—implemented as a **service principal**—so the container can authenticate to Foundry resources (models, tools/Toolbox) and other downstream Azure servic

## Different thread, different memory

In [15]:
thread_2 = {
    "configurable": {
        "thread_id": "research-thread-002"
    }
}

result_other = persistent_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What identity part was I asking about earlier?",
            }
        ]
    },
    config=thread_2,
)

print(result_other["messages"][-1].content)

[{'type': 'text', 'text': 'I can’t tell from this chat: there isn’t any earlier context here besides your question, so I don’t know which “identity part” you mean.\n\nIf you paste the earlier message (or tell me the exact phrase you used), I’ll point to the specific identity part you were asking about.', 'annotations': [], 'id': 'msg_0c2c3a4143308ea0006aa20ccdcc9081969bc62a82ce2b1eb7'}]


## Inspect Available checkpoints

In [16]:
history = list(
    persistent_agent.get_state_history(thread_1)
)

print("Number of checkpoints:", len(history))

Number of checkpoints: 8


In [18]:
for i, checkpoint in enumerate(history):
    print(f"\nCHECKPOINT {i}")
    print("created:", checkpoint.created_at)
    print("config:", checkpoint.config)


CHECKPOINT 0
created: 2026-09-10T01:47:57.397308+00:00
config: {'configurable': {'thread_id': 'research-thread-001', 'checkpoint_ns': '', 'checkpoint_id': '1f1acb9a-5b12-645e-8006-ba4f65d51fb6'}}

CHECKPOINT 1
created: 2026-09-10T01:47:27.585165+00:00
config: {'configurable': {'thread_id': 'research-thread-001', 'checkpoint_ns': '', 'checkpoint_id': '1f1acb99-3ec2-6b83-8005-37d983c77c3d'}}

CHECKPOINT 2
created: 2026-09-10T01:47:27.579164+00:00
config: {'configurable': {'thread_id': 'research-thread-001', 'checkpoint_ns': '', 'checkpoint_id': '1f1acb99-3eb4-611f-8004-e107caba373c'}}

CHECKPOINT 3
created: 2026-09-10T01:47:27.571855+00:00
config: {'configurable': {'thread_id': 'research-thread-001', 'checkpoint_ns': '', 'checkpoint_id': '1f1acb99-3ea2-6398-8003-1c3244e2df30'}}

CHECKPOINT 4
created: 2026-09-10T00:32:41.737121+00:00
config: {'configurable': {'thread_id': 'research-thread-001', 'checkpoint_ns': '', 'checkpoint_id': '1f1acaf2-2261-644f-8002-83293f486717'}}

CHECKPOINT 5
c

## Chat history is only one part of a checkpoint

A transcript store might contain:

user
assistant
user
assistant

A LangGraph checkpoint can preserve the graph's evolving state:

- messages
- node state
- pending execution state
- interrupts
- metadata
- checkpoint lineage

This is why checkpointing becomes the foundation for durable execution.

In [19]:
tables = connection.execute(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """
).fetchall()

tables

[('checkpoints',), ('writes',)]

In [21]:
for (table_name,) in tables:
    try:
        count = connection.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]

        print(table_name, count)
    except Exception as exc:
        print(table_name, ":", exc)

checkpoints 12
writes 12


In [23]:
result_3 = persistent_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Remember that for this thread we are focusing specifically "
                    "on Hosted Agent identity and runtime responsibility."
                ),
            }
        ]
    },
    config=thread_1,
)

print(result_3["messages"][-1].content[-1]['text'])

For a **Foundry Hosted Agent**, *identity* and *runtime responsibility* split cleanly between you and the platform: **at deploy time, Foundry Agent Service automatically provisions a dedicated Microsoft Entra ID “agent identity” for that hosted agent (and a dedicated endpoint)**, and at runtime your container authenticates to Foundry and downstream services *as that agent identity* (so you grant it least-privilege permissions via Entra/RBAC and avoid embedding secrets in the image). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)) Meanwhile, **Foundry owns the runtime “plumbing”**—pulling the container image, provisioning isolated session compute, routing requests, and handling lifecycle/ops like scaling and observability—while **your code owns the agent behavior** (the orchestration loop, tool usage decisions, and what it does with model outputs) and uses the provisioned identity when it needs to access protected resources. ([learn

## Why SQLite is development-only here

SQLite is excellent for:

- local learning
- notebooks
- single-process development
- easy inspection

It is not our intended Hosted Agent production checkpointer because:

- multiple Hosted Agent sessions may execute concurrently
- multiple processes may need shared state
- local container storage is not the durable enterprise database
- coordination/concurrency requirements become more complex

# What we learned

### Thread

A stable identity for one evolving LangGraph execution/conversation.

### Checkpoint

A durable snapshot of graph state.

### Checkpointer

The persistence mechanism that saves and restores checkpoints.

### SQLiteSaver

A local development implementation of that persistence mechanism.

### Key result

The Deep Agent can continue a conversation after the original Python process
has disappeared because the state lives in SQLite rather than process memory.